# 自动微分

上一章我们完成了前向传播链路的封装，并引入了计算图的概念。本章来实现它的核心机制：**自动微分**。

---

梯度是损失函数对模型参数的导数。计算梯度有三种方式：

1. **数值微分**

给每个参数加一个微小扰动 $\epsilon$，观察损失的变化：

$$
\frac{\partial L}{\partial w_i} \approx \frac{L(w_i + \epsilon) - L(w_i)}{\epsilon}
$$

数值微分的代价极高：有多少个参数，就需要做多少次前向传播。一个百万参数的模型需要运行一百万次推理，完全不可接受。

2. **符号微分**

像我们在第一部分那样，手动推导每个参数的导数公式。对简单的线性回归还好，但一旦网络有几十层、包含各种非线性变换，手推公式可能长达数页。更致命的是，每次修改网络结构，所有公式就要推倒重来。

3. **自动微分**

**自动微分**（Automatic Differentiation）利用微积分的**链式法则**，在执行计算的同时自动累积梯度。它的核心思想是：任何复杂的运算都可以拆解为简单的基本运算；每种基本运算的导数规则是固定已知的；只要记录计算的先后顺序（计算图），就能从损失值出发，沿计算图反向逐步应用链式法则，自动算出所有参数的梯度。

---

## 链式法则

链式法则的基本形式是：

$$
\frac{dL}{dx} = \frac{dL}{dp} \cdot \frac{dp}{dx}
$$

意思是：损失 $L$ 对 $x$ 的梯度，等于 $L$ 对中间量 $p$ 的梯度（**上游梯度**）乘以 $p$ 对 $x$ 的导数（**局部导数**）。

应用到我们的网络：梯度计算从损失值张量出发，沿计算图一步步向前传播。每经过一个节点，该节点只做两件事：

1. 用链式法则，把上游梯度乘以局部导数，算出父节点的梯度并累加；
2. 对所有父节点递归调用反向函数 `backward()`，继续向前传播。

整个过程只需一次前向传播加（构建计算图）一次反向遍历（自动微分），就能得出所有参数的梯度。

In [1]:
from abc import ABC, abstractmethod

import numpy as np

## 基础架构

### 张量

计算图中的每个节点都是一个张量。张量不仅保存数据，还将承担梯度计算链路的传播工作。

为此，我们新增三个属性和一个方法：

* **梯度 `grad`**：存储当前张量的梯度，即损失 $L$ 对该张量的导数。初始化为全零数组，在反向传播中累加；
* **梯度计算函数 `gradient_fn`**：一个闭包函数，封装了当前节点的链式法则计算逻辑。
  它根据自身的梯度（`self.grad`，即上游梯度），算出父节点的梯度并累加到父节点；
* **父节点集合 `parents`**：记录当前张量的直接数据来源。反向传播时，通过它找到下一个节点；
* **反向函数 `backward()`**：梯度计算的触发入口，执行梯度计算函数 `gradient_fn()` 后递归调用所有父节点的反向函数 `backward()`。

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础层

In [3]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [4]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

## 数据

### 特征、标签

In [5]:
feature = Tensor([28.1, 58.0])
label = Tensor([165])

## 模型

### 线性层

线性层的推理函数 `forward()` 是本章改动最大的地方。除了计算预测值，现在还要在预测值张量中添加**梯度计算函数 `gradient_fn()`** 来计算权重和偏置的梯度。

线性层计算 $p = x \cdot W^T + b$，各参数的局部导数为：

$$
\frac{\partial p}{\partial W} = x, \quad
\frac{\partial p}{\partial b} = 1
$$

应用链式法则，将上游梯度 `p.grad` 与各局部导数相乘：

$$
\frac{dL}{dW} = \frac{dL}{dp} \cdot x, \quad
\frac{dL}{db} = \frac{dL}{dp}
$$

预测值张量的梯度计算函数 `gradient_fn()` 会被反向函数 `backward()` 自动调用。

In [6]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad * x.data
            self.bias.grad += np.sum(p.grad)

        p.gradient_fn = gradient_fn
        return p

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 均方误差损失函数

损失函数是计算图的终点，也是反向传播的起点。它的梯度计算函数 `gradient_fn()` 产生第一个上游梯度（单样本）：

$$
L = \sum(y - p)^2
\quad \Rightarrow \quad
\frac{dL}{dp} = -2(y - p)
$$

预测值 `p` 被放入父节点集合 `parents`，连通计算图的最后一笔。

In [7]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 训练

### 建模

In [8]:
layer = Linear(2, 1)
loss_fn = MSELoss()
print(layer)

Linear[weight(1, 2); bias(1,)]


### 前向传播

前向传播过程中，每一步计算都会同时往输出张量里注入梯度计算函数 `gradient_fn()` 和父节点集合 `parents`，动态地把计算图构建起来。

In [9]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([43.05])


### 计算损失

调用损失函数时，计算图的最后一条边也被连接上了，计算图构建完成。

In [10]:
loss = loss_fn(prediction, label)
print(f'loss: {loss}')

loss: Tensor(14871.802500000002)


此刻计算图已经建立完毕。

权重和偏置作为**叶节点**直接参与计算，但不在父节点集合 `parents` 路径上。它们的梯度在梯度计算函数 `gradient_fn()` 中被直接计算，不需要继续向前追溯。

```{figure} images/compu-graph.png
:align: center
:width: 300px
**图例：本章的计算图结构**

* $x$：特征值张量（起点）
* $p$：预测值张量（parents 链路上的中间节点）
* $w, b$：权重张量和偏置张量（叶节点，不在 parents 链路上）
* $l$：损失值张量（终点，backward() 的调用入口）

```

### 梯度计算

只需在损失值张量上调用一次反向函数 `backward()`，梯度就会沿计算图自动反向传播，所有参数的梯度都会被计算并存入各自的梯度 `grad` 中。

In [11]:
loss.backward()
print(f'weight.grad: {layer.weight.grad}')
print(f'bias.grad:   {layer.bias.grad}')

weight.grad: [[ -6853.59 -14146.2 ]]
bias.grad:   [-243.9]


### 反向传播

梯度已计算完毕，根据梯度更新参数。

In [12]:
layer.weight.data -= layer.weight.grad
layer.bias.data -= layer.bias.grad
print(f'weight: {layer.weight}')
print(f'bias:   {layer.bias}')

weight: Tensor([[ 6854.09 14146.7 ]])
bias:   Tensor([243.9])


## 验证

现在，让我们用新的权重和偏置再预测一次。

### 推理

In [13]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([1013352.429])


### 评估

In [14]:
loss = loss_fn(prediction, label)
print(f'loss: {loss}')

loss: Tensor(1026548766283.6302)


损失值爆炸，但是这符合预期（没有使用学习率）。

## 课后练习

在 `loss.backward()` 调用前后分别打印 `layer.weight.grad`，观察梯度如何从全零变化。